## **#Importing necessary libraries for data manipulation**

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# **#Creating DataFrames**

In [ ]:
orders = pd.read_csv("../content/drive/MyDrive/ADA2026_Instacart_Project/data/raw/instacart/orders.csv")
products = pd.read_csv("../content/drive/MyDrive/ADA2026_Instacart_Project/data/raw/instacart/products.csv")
departments= pd.read_csv("../content/drive/MyDrive/ADA2026_Instacart_Project/data/raw/instacart/departments.csv")
orderprodprior= pd.read_csv("../content/drive/MyDrive/ADA2026_Instacart_Project/data/raw/instacart/order_products__prior.csv")

# **TASK 1**

---



# **#Constructing Transaction Dataset**

In [ ]:
# Merge to get product names
df = orderprodprior.merge(products[['product_id', 'product_name']], on='product_id', how='left')
df.rename(columns={'product_name': 'ProductName'}, inplace=True)

print("Transaction dataset successfully created")
print(f"  Total rows: {len(df)}")
print(f"  Total orders: {df['order_id'].nunique()}")
print(f"  Total products: {df['ProductName'].nunique()}")

# Save for next step
df.to_csv('transaction_dataset.csv', index=False)

Transaction dataset successfully created
  Total rows: 32434489
  Total orders: 3214874
  Total products: 49677


* Why Merge: Raw transaction tables (like order_products__prior) only contain numerical IDs. Merging with the products table allows the analysis to use Product Names (e.g Bananas) instead of abstract IDs (e.g 24852).

# **#Applying Filter**

In [ ]:
# Load transaction dataset
df = pd.read_csv('transaction_dataset.csv')

# Filter for Top 100 Products
print("Applying filter for top 100 products")
product_counts = df['ProductName'].value_counts()
top_products = product_counts.head(100).index.tolist()
filtered_df = df[df['ProductName'].isin(top_products)].copy()

print(f" Filtering applied")
print(f"  Original rows: {len(df)}")
print(f"  Filtered rows: {len(filtered_df)}")
print(f"  Unique products in top 100: {filtered_df['ProductName'].nunique()}")

# Save filtered dataset
filtered_df.to_csv('filtered_transactions.csv', index=False)
product_counts.head(100).to_csv('top_100_products.csv')

Applying filter for top 100 products
 Filtering applied
  Original rows: 32434489
  Filtered rows: 7483881
  Unique products in top 100: 100


* Why? Identifying associations between low-volume items would provide little return on investment (ROI).

# **#Show Summary**

In [ ]:
# Load filtered dataset
filtered_df = pd.read_csv('filtered_transactions.csv')
product_counts = pd.read_csv('top_100_products.csv', index_col=0).squeeze()

# Calculate Summary Statistics
unique_orders = filtered_df['order_id'].nunique()
basket_sizes = filtered_df.groupby('order_id').size()
avg_basket_size = basket_sizes.mean()
most_frequent_product = product_counts.index[0]

# Display Results
print("TASK 1: DATA PREPARATION - SUMMARY OUTPUTS")
print("-"*70)

print("\n KEY RESULTS:")
print("-"*70)
print(f"Total number of unique orders in filtered dataset: {unique_orders}")
print(f" Average basket size for these top 100 items: {avg_basket_size:.2f}")
print(f" Name of #1 most frequent product: {most_frequent_product}")

print(f"\n ADDITIONAL STATISTICS:")
print("-"*70)
print(f"Total transactions: {len(filtered_df)}")
print(f"Unique products (top 100): {filtered_df['ProductName'].nunique()}")
print(f"Min basket size: {basket_sizes.min()}")
print(f"Max basket size: {basket_sizes.max()}")

print(f"\n TOP 10 PRODUCTS:")
print("-"*70)
for rank, (product, count) in enumerate(product_counts.head(10).items(), 1):
    print(f"{rank:2d}. {product:40s} - {count:5.0f} occurrences")

print("\n Data Preparation Complete")
print(""*70)

# Prepare transaction list for Apriori
transactions = filtered_df.groupby('order_id')['ProductName'].apply(lambda x: x.unique().tolist()).tolist()

print(f"\nReady for Apriori Algorithm with {len(transactions)} transactions")

TASK 1: DATA PREPARATION - SUMMARY OUTPUTS
----------------------------------------------------------------------

 KEY RESULTS:
----------------------------------------------------------------------
Total number of unique orders in filtered dataset: 2351240
 Average basket size for these top 100 items: 3.18
 Name of #1 most frequent product: Banana

 ADDITIONAL STATISTICS:
----------------------------------------------------------------------
Total transactions: 7483881
Unique products (top 100): 100
Min basket size: 1
Max basket size: 28

 TOP 10 PRODUCTS:
----------------------------------------------------------------------
 1. Banana                                   - 472565 occurrences
 2. Bag of Organic Bananas                   - 379450 occurrences
 3. Organic Strawberries                     - 264683 occurrences
 4. Organic Baby Spinach                     - 241921 occurrences
 5. Organic Hass Avocado                     - 213584 occurrences
 6. Organic Avocado               

# **TASK 2**

---



# **#Apriori with multiple thresholds**

In [ ]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori

# Step 1: Encode the transactions list from Task 1 into a sparse matrix
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
basket_df = pd.DataFrame(te_array, columns=te.columns_)

# Step 2: Run Apriori with Threshold #1 (1% Support)
# This identifies items appearing in at least 23,512 orders
support_1 = 0.01
itemsets_1 = apriori(basket_df, min_support=support_1, use_colnames=True)
count_1 = len(itemsets_1)

# Step 3: Run Apriori with Threshold #2 (0.5% Support)
# This identifies items appearing in at least 11,756 orders
support_2 = 0.005
itemsets_2 = apriori(basket_df, min_support=support_2, use_colnames=True)
count_2 = len(itemsets_2)

print("--- TASK 2: PARAMETER ANALYSIS RESULTS ---")
print(f"Count of frequent itemsets at {support_1*100}% support: {count_1}")
print(f"Count of frequent itemsets at {support_2*100}% support: {count_2}")

--- TASK 2: PARAMETER ANALYSIS RESULTS ---
Count of frequent itemsets at 1.0% support: 130
Count of frequent itemsets at 0.5% support: 271


* 1%: Allows to capture items that are commercially significant but may only yield trivial rules.

* 0,5%: Allows to capture often surprising, relationships between items that are bought together by a smaller segment of customers, rather than mass-market pairings like bread and butter.

# **TASK 3**

---

# **#Filter using lift and confidence**

In [ ]:
from mlxtend.frequent_patterns import association_rules

# Step 1: Use the itemsets from the 0.5% support run
# frequent_itemsets_2 was the variable for the 271 itemsets
rules = association_rules(itemsets_2, metric="lift", min_threshold=1.0)

# Step 2: Filter for high-quality rules
filtered_rules = rules[rules['confidence'] >= 0.3]

# Step 3: Sort by Lift to find the most interesting associations
top_10_rules = filtered_rules.sort_values(by='lift', ascending=False).head(10)

# Display for Task 3 output
print("--- TASK 3: TOP 10 ASSOCIATION RULES ---")
print(top_10_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

--- TASK 3: TOP 10 ASSOCIATION RULES ---
                                      antecedents               consequents  \
308  (Organic Hass Avocado, Organic Strawberries)  (Bag of Organic Bananas)   
52                         (Organic Navel Orange)  (Bag of Organic Bananas)   
49         (Organic Large Extra Fancy Fuji Apple)  (Bag of Organic Bananas)   
113                          (Organic Fuji Apple)                  (Banana)   
80                               (Bartlett Pears)                  (Banana)   
96                             (Honeycrisp Apple)                  (Banana)   
87                               (Broccoli Crown)                  (Banana)   
93                               (Cucumber Kirby)                  (Banana)   
91                             (Clementines, Bag)                  (Banana)   
134                         (Seedless Red Grapes)                  (Banana)   

      support  confidence      lift  
308  0.005864    0.348580  2.129825  
52   0.005784

# TASK 5

---



In [ ]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# Merge department names into Task 1 filtered data
df_dept = filtered_df.merge(products[['product_name', 'department_id']], left_on='ProductName', right_on='product_name')
df_dept = df_dept.merge(departments, on='department_id')

# Construct Department-level baskets
dept_basket = df_dept.groupby('order_id')['department'].apply(lambda x: list(set(x)))

# Run Apriori at the Category Level
te_dept = TransactionEncoder()
te_dept_array = te_dept.fit(dept_basket).transform(dept_basket)
dept_basket_df = pd.DataFrame(te_dept_array, columns=te_dept.columns_)

# Use 1% support (0.01) to find more department-level associations.
frequent_depts = apriori(dept_basket_df, min_support=0.01, use_colnames=True)
dept_rules = association_rules(frequent_depts, metric="lift", min_threshold=1.0)

# Output Top 5 Rules for Analysis
print("--- TASK 5: TOP CATEGORY-LEVEL RULES ---")
print(dept_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].sort_values(by='lift', ascending=False).head(5))

--- TASK 5: TOP CATEGORY-LEVEL RULES ---
              antecedents            consequents   support  confidence  \
12                 (deli)  (produce, dairy eggs)  0.017197    0.366928   
9   (produce, dairy eggs)                 (deli)  0.017197    0.068937   
8         (produce, deli)           (dairy eggs)  0.017197    0.425684   
13           (dairy eggs)        (produce, deli)  0.017197    0.054517   
3            (dairy eggs)                 (deli)  0.018862    0.059797   

        lift  
12  1.470893  
9   1.470893  
8   1.349492  
13  1.349492  
3   1.275884  


*  From a management perspective:

-Product rules are best for digital recommendations and bundling. WHILE

-Category rules are superior for physical store layout and aisle positioning to maximize traffic flow between high lift departments like Deli and Produce.